In [1]:
import pandas as pd

In [2]:
sf = pd.read_csv('spike_forensics.csv', dtype={'period': str})

print(sf.shape)
print(sf.dtypes)
display(sf.head())

(853, 5)
period          object
window_group    object
event_name      object
n_events         int64
n_users          int64
dtype: object


,period,window_group,event_name,n_events,n_users
0,20180628,pre_window,session_start,1113,730
1,20180628,pre_window,user_engagement,19919,779
2,20180628,pre_window,post_score,2410,331
3,20180628,pre_window,level_end,761,107
4,20180628,pre_window,level_end_quickplay,5751,560


In [3]:
spike_dates = ['20180618', '20180619', '20180625', '20180626', '20180628', '20180630', '20180701']
interleaved_dates = ['20180620', '20180621', '20180622', '20180623', '20180624', '20180627', '20180629']

label_map = {}

for date in spike_dates:
    label_map[date] = 'spike'

for date in interleaved_dates:
    label_map[date] = 'interleaved'

label_map['stable_block'] = 'anchor'

sf['label'] = sf['period'].map(label_map)

sf

,period,window_group,event_name,n_events,n_users,label
0,20180628,pre_window,session_start,1113,730,spike
1,20180628,pre_window,user_engagement,19919,779,spike
2,20180628,pre_window,post_score,2410,331,spike
3,20180628,pre_window,level_end,761,107,spike
4,20180628,pre_window,level_end_quickplay,5751,560,spike
...,...,...,...,...,...,...
848,stable_block,pre_window,notification_foreground,1,1,anchor
849,stable_block,in_window,dynamic_link_app_open,13,6,anchor
850,stable_block,in_window,challenge_accepted,3,2,anchor
851,stable_block,pre_window,dynamic_link_app_open,3,2,anchor


In [4]:
print(sf['label'].isna().sum())


sf.groupby('label')['period'].nunique()

0


,period
label,
anchor,1
interleaved,7
spike,7


In [5]:
pre_window = sf[sf['window_group'] == 'pre_window']

grouped_by_label_event = pre_window.groupby(['label', 'event_name'])['n_events'].sum().reset_index()
grouped_by_label_event['total'] = grouped_by_label_event.groupby('label')['n_events'].transform('sum')
grouped_by_label_event['share'] = grouped_by_label_event['n_events'] / grouped_by_label_event['total']

pivoted = grouped_by_label_event.pivot_table(index='event_name', columns='label', values='share')
pivoted.sort_values('spike', ascending=False)

label,anchor,interleaved,spike
event_name,,,
user_engagement,2.243122e-01,0.228853,0.394687
level_start_quickplay,9.743128e-02,0.095897,0.165786
level_end_quickplay,6.466267e-02,0.063530,0.115223
screen_view,4.193910e-01,0.417086,0.061335
post_score,4.346699e-02,0.041599,0.057895
level_fail_quickplay,2.411953e-02,0.025910,0.046327
level_reset_quickplay,2.344017e-02,0.021132,0.035571
level_start,5.719341e-03,0.007858,0.022215
session_start,1.299960e-02,0.011337,0.020994
